# Retail shoplifting model training on GPU

Train the remaining expanded 2D BiGRU experiment: 100 normal RetailS pose files, seeds 0/1/2, 15 epochs each. The five matched A–E runs are already preserved locally. This notebook uses the same prepared features, masks, source-video split, architecture and normal-validation p95 threshold. GPU arithmetic can differ from CPU results.

**Before running:** select **Runtime → Change runtime type → GPU** (for example T4). Upload `Retail_Shoplifting_Colab_Training.zip` to the top level of **My Drive**. Run cells in order. Keep the Drive output directory to resume after a disconnect.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
ZIP = Path("/content/drive/MyDrive/Retail_Shoplifting_Colab_Training.zip")
OUT = Path("/content/drive/MyDrive/Retail_Shoplifting_Training_v1")
ROOT = Path("/content/retail_training")
assert ZIP.is_file(), f"Upload the training ZIP to My Drive: {ZIP}"

## 1. Extract and verify inputs
Check: verification reports 41,841 normal windows, a disjoint source-video split, and 1,856 retained test windows. The ZIP contains prepared pose tensors, not RGB videos. It preserves the current RetailS-specific coordinate interpretation and quality gates; this does not establish pose or detection accuracy.

In [ ]:
import zipfile, hashlib, json
with zipfile.ZipFile(ZIP) as archive:
    for entry in archive.infolist():
        target = (ROOT / entry.filename).resolve()
        assert target.is_relative_to(ROOT.resolve()), "Unsafe archive path"
    archive.extractall(ROOT)
manifest = json.loads((ROOT / "bundle_manifest.json").read_text())
for name, expected in manifest["files"].items():
    with (ROOT / name).open("rb") as f:
        assert hashlib.file_digest(f, "sha256").hexdigest() == expected, name
print("Integrity check passed")
print(json.dumps({"split": manifest["split"], "counts": {k: v["counts"] for k,v in manifest["datasets"].items()}}, indent=2))

## 2. Check the GPU and dependencies
Check: CUDA must be available. Keep Colab’s installed PyTorch build; the notebook only installs missing supporting packages.

In [ ]:
import importlib.util, subprocess, sys
for module, package in [("numpy", "numpy"), ("sklearn", "scikit-learn"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
import torch
assert torch.cuda.is_available(), "Select a GPU runtime, reconnect, then rerun these cells"
print("PyTorch:", torch.__version__, "GPU:", torch.cuda.get_device_name(0))

## 3. Train three seeds
Check: each seed reaches epoch 15/15 and produces `complete.json`. Best weights are selected using normal validation loss. Test labels are used only for evaluation. Checkpoints are saved to Drive after each epoch; rerun this cell to resume at the next saved epoch. A disconnect can lose the current unfinished epoch.

The stopped local runs did not save optimizer/RNG state, so these GPU runs start afresh. Completed Colab epochs include that state. Keep the same GPU/runtime software where possible; changing hardware can change floating-point results.

In [ ]:
subprocess.run([sys.executable, "-u", str(ROOT / "scripts/train_colab.py"), "--data-root", str(ROOT), "--output", str(OUT), "--device", "cuda", "--seeds", "0", "1", "2"], check=True)

## 4. Verify and inspect results
Check: three completed runs, 15 logged epochs per run, finite metrics. Precision alone can be misleading because the retained test windows contain many anomalies. Report recall, F1, ROC-AUC, PR-AUC and rejected-window coverage as well. These are video-mask-derived anomaly labels, not person-level theft annotations.

In [ ]:
import pandas as pd
rows = []
for seed in (0, 1, 2):
    folder = OUT / f"expanded_a_seed{seed}"
    complete = json.loads((folder / "complete.json").read_text())
    for name, expected in complete["files"].items():
        with (folder / name).open("rb") as f:
            assert hashlib.file_digest(f, "sha256").hexdigest() == expected, name
    assert len(json.loads((folder / "history.json").read_text())) == 15
    report = json.loads((folder / "evaluation.json").read_text())
    rows.append({"seed": seed, "best_epoch": report["best_epoch"], **report["metrics"]["combined"]})
table = pd.DataFrame(rows)
display(table)
display(table[["precision", "recall", "f1", "roc_auc", "pr_auc"]].agg(["mean", "std"]))

## 5. Export results for the local repository
Download the results ZIP, place it in the local repository’s `outputs/colab_handoff/` directory, and follow `COLAB_TRAINING.md`. Keep Drive checkpoints until local validation is complete. The export includes the CPU-loadable best weights, scores, normal calibration errors, logs, provenance and checksums. Full resume states remain on Drive.

In [ ]:
RESULT = OUT.parent / "Retail_Shoplifting_Colab_Results.zip"
with zipfile.ZipFile(RESULT, "w", zipfile.ZIP_DEFLATED) as archive:
    archive.write(OUT / "bundle_manifest.json", "bundle_manifest.json")
    for subset in ("train", "test_realworld", "test_staged"):
        archive.write(ROOT / f"data/{subset}.json", f"data/{subset}.json")
    for seed in (0, 1, 2):
        folder = OUT / f"expanded_a_seed{seed}"
        complete = json.loads((folder / "complete.json").read_text())
        for name in ["complete.json", *complete["files"]]:
            archive.write(folder / name, f"{folder.name}/{name}")
print(RESULT, round(RESULT.stat().st_size / 1024**2, 1), "MiB")
from google.colab import files
files.download(str(RESULT))